# NBA Shot Quality — from tracking features to a calibrated model & Points Over Expectation

This notebook puts the **NBA Shot Quality Features (2015-16 SportVU)** dataset to work end to end:

1. **Explore** where shots come from and how make rate varies with distance, defender pressure, and court location.
2. **Train a calibrated shot-make-probability model** (XGBoost) with a *game-disjoint* split so no game leaks between train and test.
3. **Evaluate calibration first** (Brier, log-loss, reliability diagram) against constant and distance-only baselines.
4. **Turn probabilities into a metric** — out-of-fold **Points Over Expectation (POE)** — and rank the best and worst shot-makers.
5. **Visualise** a player's shot chart coloured by POE.

Every feature is measured **at the moment of release**, so the model conditions on shot context, not on shooter identity.

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import patches

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

# Locate the CSV — works on Kaggle (/kaggle/input/...) and locally.
cands = glob.glob("/kaggle/input/**/*.csv", recursive=True)
CSV = next((p for p in cands if "shot" in os.path.basename(p).lower()), (cands[0] if cands else None))
if CSV is None:
    CSV = "data/shot_features_valid2.csv"   # local fallback
print("Loading:", CSV)

df = pd.read_csv(CSV)
print(f"{len(df):,} shots | {df['game_id'].nunique()} games | "
      f"{df['player_name'].nunique()} shooters | {df['team_id'].nunique()} teams")
df.head()

## 1. What's in the data

One row per field-goal attempt. A quick profile of the outcome and shot mix:

In [ ]:
overview = {
    "shots": len(df),
    "games": df["game_id"].nunique(),
    "shooters": df["player_name"].nunique(),
    "league make rate": round(df["made_shot"].mean(), 3),
    "3PT share (PBP)": round(df["description"].fillna("").str.contains("3PT", case=False).mean(), 3),
    "dunk/tip share": round(df["is_dunk_or_tip"].mean(), 3),
}
pd.Series(overview)

## 2. Exploratory analysis

### Make rate and volume by distance
The classic shot-quality gradient: efficiency is high at the rim, collapses in the mid-range, and ticks back up beyond the arc (where each make is worth 3).

In [ ]:
bins   = [0, 4, 8, 12, 16, 20, 23.75, 30, 50]
labels = ["0-4", "4-8", "8-12", "12-16", "16-20", "20-23.75", "23.75-30", "30+"]
df["dist_bin"] = pd.cut(df["dist"], bins=bins, labels=labels, include_lowest=True)
by_dist = df.groupby("dist_bin")["made_shot"].agg(["mean", "count"])

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar(by_dist.index.astype(str), by_dist["mean"], color="#1f4e79")
ax[0].set_title("Make rate by shot distance"); ax[0].set_ylabel("FG%"); ax[0].tick_params(axis="x", rotation=45)
ax[1].bar(by_dist.index.astype(str), by_dist["count"], color="#c0504d")
ax[1].set_title("Shot volume by distance"); ax[1].set_ylabel("shots"); ax[1].tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

### Make rate by defender pressure
Closer contests suppress efficiency — one of the signals the model will lean on beyond raw distance.

In [ ]:
df["def_bin"] = pd.cut(df["closest_def_dist"], [0, 2, 4, 6, 8, 100],
                       labels=["0-2 very tight", "2-4 tight", "4-6 open", "6-8", "8+ wide open"])
df.groupby("def_bin")["made_shot"].agg(make_rate="mean", shots="count").round(3)

### Make rate by court location
Coordinates are normalised to the right-hand basket offensive half (94×50 ft court). Each hexbin cell is coloured by empirical make rate.

In [ ]:
def draw_half_court(ax, color="black", lw=1.2):
    ax.add_patch(patches.Rectangle((47, 0), 47, 50, fill=False, ec=color, lw=lw))   # boundary
    ax.add_patch(patches.Rectangle((69, 17), 19, 16, fill=False, ec=color, lw=lw))  # paint
    ax.add_patch(patches.Circle((88.75, 25), 0.75, fill=False, ec=color, lw=lw))    # rim
    ax.add_patch(patches.Arc((88.75, 25), 8, 8, theta1=90, theta2=270, color=color, lw=lw))  # restricted
    ax.add_patch(patches.Arc((69, 25), 12, 12, color=color, lw=lw))                 # FT circle
    r = 23.75; x0 = 88.75 - np.sqrt(r**2 - 22**2)
    ax.plot([94, x0], [3, 3], color=color, lw=lw); ax.plot([94, x0], [47, 47], color=color, lw=lw)
    th = np.degrees(np.arccos(22 / r))
    ax.add_patch(patches.Arc((88.75, 25), 2*r, 2*r, theta1=180-th, theta2=180+th, color=color, lw=lw))
    ax.set_xlim(47, 94); ax.set_ylim(0, 50); ax.set_aspect("equal"); ax.axis("off")

sub = df[df["x"].between(47, 94) & df["y"].between(0, 50)]
fig, ax = plt.subplots(figsize=(8, 7))
hb = ax.hexbin(sub["x"], sub["y"], C=sub["made_shot"], gridsize=30, cmap="coolwarm", mincnt=20)
draw_half_court(ax)
fig.colorbar(hb, ax=ax, label="make rate")
ax.set_title("Make rate by court location (min 20 shots / cell)")
plt.show()

## 3. A leakage-safe train/test split

Shots from the same game are correlated (same shooters, same defenders, same night). Splitting at the **shot** level would leak that structure. We split at the **game** level with `GroupShuffleSplit` on `game_id`, and carve a validation slice off the training games for early stopping.

We drop identifiers and post-hoc flags from the feature set (`player_name`, `game_id`, `description`, `is_3_pointer`, `is_dunk_or_tip`, ...), leaving the release-frame features.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

META = ["player_name", "game_time", "quarter", "score_margin", "description",
        "team_id", "game_id", "is_dunk_or_tip", "is_3_pointer", "closest_def_name",
        "dist_bin", "def_bin"]
TARGET = "made_shot"
features = [c for c in df.columns if c not in META + [TARGET]]
X, y, groups = df[features], df[TARGET], df["game_id"]
print(len(features), "features:", features)

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
tr, te = next(gss.split(X, y, groups))
Xtr, Xte, ytr, yte, gtr = X.iloc[tr], X.iloc[te], y.iloc[tr], y.iloc[te], groups.iloc[tr]
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
tr2, va = next(gss2.split(Xtr, ytr, gtr))
Xt, Xv, yt, yv = Xtr.iloc[tr2], Xtr.iloc[va], ytr.iloc[tr2], ytr.iloc[va]
assert set(gtr.iloc[va]).isdisjoint(set(groups.iloc[te]))
print(f"train {len(Xt):,} | val {len(Xv):,} | test {len(Xte):,}")

## 4. Train a calibrated XGBoost model

Gradient-boosted trees handle the mixed, non-linear, NaN-containing tabular features well. We optimise **log-loss** (a proper scoring rule) with early stopping on the validation set — the goal is well-calibrated probabilities, not threshold accuracy.

In [ ]:
import xgboost as xgb

model = xgb.XGBClassifier(
    n_estimators=2000, learning_rate=0.02, max_depth=5,
    min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
    objective="binary:logistic", eval_metric="logloss",
    early_stopping_rounds=50, random_state=RANDOM_STATE, n_jobs=-1,
)
model.fit(Xt, yt, eval_set=[(Xv, yv)], verbose=False)
print("best iteration:", model.best_iteration)
p_test = model.predict_proba(Xte)[:, 1]

## 5. Evaluation — calibration first

We compare the full model against two references on the identical held-out test games:
- **Constant** — always predict the league base rate.
- **Distance-only logistic** — the classic "expected FG% from distance" baseline.

Lower Brier / log-loss and higher AUC are better.

In [ ]:
from sklearn.metrics import (brier_score_loss, log_loss, roc_auc_score,
                             average_precision_score, accuracy_score)
from sklearn.linear_model import LogisticRegression

def metrics(y_true, p):
    return dict(Brier=brier_score_loss(y_true, p),
                LogLoss=log_loss(y_true, p, labels=[0, 1]),
                ROC_AUC=roc_auc_score(y_true, p),
                PR_AUC=average_precision_score(y_true, p),
                Acc=accuracy_score(y_true, (p >= 0.5).astype(int)))

base   = np.full(len(yte), yt.mean())
distlr = LogisticRegression().fit(Xt[["dist"]], yt)
p_dist = distlr.predict_proba(Xte[["dist"]])[:, 1]

pd.DataFrame({
    "Constant (base rate)":   metrics(yte, base),
    "Distance-only logistic": metrics(yte, p_dist),
    "XGBoost (all features)": metrics(yte, p_test),
}).T.round(4)

In [ ]:
from sklearn.calibration import calibration_curve

pt, pp = calibration_curve(yte, p_test, n_bins=12, strategy="quantile")
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].plot([0, 1], [0, 1], "--", color="gray", label="perfectly calibrated")
ax[0].plot(pp, pt, "o-", color="#1f4e79", label="XGBoost")
ax[0].set_xlabel("mean predicted P(make)"); ax[0].set_ylabel("empirical make rate")
ax[0].set_title(f"Reliability diagram (Brier = {brier_score_loss(yte, p_test):.3f})"); ax[0].legend()
ax[1].hist(p_test[yte == 0], bins=40, alpha=.55, label="misses", color="#c0504d")
ax[1].hist(p_test[yte == 1], bins=40, alpha=.55, label="makes",  color="#1f4e79")
ax[1].set_xlabel("predicted P(make)"); ax[1].set_title("Predicted probability by outcome"); ax[1].legend()
plt.tight_layout(); plt.show()

### What the model leans on
Distance dominates, followed by defender pressure and release mechanics/kinematics — with the behavioural archetype probabilities contributing on top.

In [ ]:
imp = pd.Series(model.feature_importances_, index=features).sort_values()
fig, ax = plt.subplots(figsize=(8, 8))
imp.tail(20).plot.barh(ax=ax, color="#1f4e79")
ax.set_title("Top 20 features by gain"); ax.set_xlabel("relative importance")
plt.tight_layout(); plt.show()

### Where the model is confident — per-zone breakdown
The restricted area is easy (bimodal: uncontested dunks vs contested rim attempts); the mid-range is the hardest regime.

In [ ]:
def zone(d):
    if d <= 4:  return "Restricted area"
    if d <= 14: return "Paint (non-RA)"
    if d < 22:  return "Mid-range"
    return "3-pointer"

z = Xte.assign(_y=yte.values, _p=p_test)
z["zone"] = z["dist"].map(zone)
rows = []
for zz in ["Restricted area", "Paint (non-RA)", "Mid-range", "3-pointer"]:
    s = z[z.zone == zz]
    if len(s):
        rows.append(dict(zone=zz, n=len(s), make_rate=s._y.mean(),
                         Brier=brier_score_loss(s._y, s._p),
                         ROC_AUC=roc_auc_score(s._y, s._p) if s._y.nunique() == 2 else np.nan))
pd.DataFrame(rows).round(3)

## 6. Points Over Expectation (POE)

With a calibrated P(make), a shot's value over an average shooter in the same situation is:

$$\text{POE} = v \cdot (y - \hat p)$$

where $v \in \{2,3\}$ is the shot's point value, $y$ the outcome, and $\hat p$ the predicted make probability. To avoid flattering players on shots the model trained on, we predict **out-of-fold** (GroupKFold over games), then sum POE per player.

In [ ]:
from sklearn.model_selection import GroupKFold

oof = np.full(len(X), np.nan)
gkf = GroupKFold(n_splits=5)
for k, (tri, vai) in enumerate(gkf.split(X, y, groups), 1):
    m = xgb.XGBClassifier(
        n_estimators=400, learning_rate=0.02, max_depth=5,
        min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
        objective="binary:logistic", eval_metric="logloss",
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    m.fit(X.iloc[tri], y.iloc[tri], verbose=False)
    oof[vai] = m.predict_proba(X.iloc[vai])[:, 1]
    print(f"fold {k}/5 done")

value = np.where(df["description"].fillna("").str.contains("3PT", case=False), 3, 2)
poe = pd.DataFrame({"player_name": df["player_name"], "x": df["x"], "y": df["y"],
                    "made": y.values, "xmake": oof, "value": value})
poe["POE"] = poe["value"] * (poe["made"] - poe["xmake"])
print("OOF Brier:", round(brier_score_loss(poe['made'], poe['xmake']), 4))

In [ ]:
lb = (poe.groupby("player_name")
         .agg(shots=("made", "size"), total_POE=("POE", "sum"))
         .reset_index())
lb = lb[lb["shots"] >= 150].copy()
lb["POE_per_100"] = 100 * lb["total_POE"] / lb["shots"]
lb = lb.sort_values("total_POE", ascending=False)

print("TOP 10 — most points OVER expectation (min 150 shots)")
display(lb.head(10).round(1).reset_index(drop=True))
print("\nBOTTOM 10 — most points UNDER expectation")
display(lb.tail(10).round(1).iloc[::-1].reset_index(drop=True))

## 7. A player's shot chart, coloured by POE

Blue = made a hard shot (positive POE); red = missed an easy one (negative POE). By default we plot the top player from the leaderboard.

In [ ]:
who = lb.iloc[0]["player_name"]
pdf = poe[poe["player_name"] == who]
vmax = max(0.5, np.nanpercentile(np.abs(pdf["POE"]), 95))

fig, ax = plt.subplots(figsize=(8, 7))
draw_half_court(ax)
sc = ax.scatter(pdf["x"], pdf["y"], c=pdf["POE"], cmap="coolwarm_r",
                vmin=-vmax, vmax=vmax, s=35, edgecolors="white", linewidths=.4, zorder=3)
fig.colorbar(sc, ax=ax, label="POE per shot")
ax.set_title(f"{who} — {len(pdf)} shots — total POE {pdf['POE'].sum():+.0f}")
plt.show()

## Takeaways & next steps

- A portable, release-frame feature set yields a **well-calibrated** shot-make model that clears distance-only baselines on games it never saw.
- The same calibrated probability powers **POE**, a defensible shot-making metric, and per-zone / per-player diagnostics.
- **Try next:** tune hyperparameters; add a defender-POE (nearest-defender) view; aggregate POE by 5-man lineup; compare corner vs above-the-break threes; or test how much accuracy the archetype features add via ablation.

*Features are measured pre-release and the split is game-disjoint, so results reflect genuine shot-context signal rather than shooter-identity memorisation.*